# Gilbert Owen

# 2802432220

# DTSC6008001 – Text Mining

# 1. Data Collection

### A. Objective

**Target**: Our target is 5000 news from 5 specific news domain.

**Use** : "finance.detik.com" for finance, "inet.detik.com" for technology, "sport.detik.com" for sports, "health.detik.com" for health, "hot.detik.com" for entertainment.

**Collect via** : Web Scrapping

**Constraint** : Records containing the main article text

**Disclaimer** : data source, scraping method/tools, data collection period, number of collected records, data fields are on section 11.





### B. Data Source

**Site :** the website that is gonna be target of scrapping is **[ detik.com ]** - It is a major Indonesian news portal which has channel subdomains that is organized, also ready-made category labels

**Why Detik.com :** We choose Detik.com because it doesnt aggresively target or forbid than the other that heavily enforce automated bot bans. Also, its robot.txt permits crawling by (`Allow: /`) on every channel that we used.

**Website and its label :** 

"finance.detik.com" for finance

"inet.detik.com" for technology

"sport.detik.com" for sports

"health.detik.com" for health"

"hot.detik.com" for entertainment.

### C. How Is The Structure of Detik.com

**AI USED**

**Index page** (list of articles for one day):
`https://<channel>/indeks?date=YYYY-MM-DD&page=N` — 20 articles per page; page 1's
pagination links tell you the last page for that day.

**Article URL:** `https://<channel>/<section>/d-<id>/<slug>`
`<id>` is a unique numeric ID, used to deduplicate. `<section>` names like `foto-news`,
`video`, `detiktv`, `grafis` are photo/video/infographic pages with little or no article
text, so they are skipped.

**Article page:** the body is every `<p>` inside `div.detail__body-text`. Title, date,
author and tags come from `<meta>` tags (`og:title`, `dtk:publishdate`, `dtk:author`,
`dtk:keywords`) — these are the same across Detik's two page templates, unlike the visible
HTML.

**Known quirk:** the index page sometimes serves a different day's page than requested
(caching issue). The fix: if a requested page returns no article IDs you haven't already
seen, request it again.

### D. Import Libraries

In [1]:
import random, re, time
from datetime import timedelta, date
from pathlib import Path
import pandas as pd
import requests
from bs4 import BeautifulSoup
from tqdm.auto import tqdm

d:\anaconda\envs\deep-learning\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### E. Helper Functions

There're functions that are going to be used to help us such as :
- fetch, it takes (a URL) and returns the URL page that is also parsed by Beautiful Soup
- collect_urls, it takes a channel base URL and one date. The function returns list of that one date article URLs
- parse_article, it takes one article URL and returns a dictionary consisting of url, title, date, author, tags, and content.


### F. The code to fetch url

In [2]:
HEADERS = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                         "(KHTML, like Gecko) Chrome/128.0 Safari/537.36"}
SKIP_SECTIONS = re.compile(r"foto|video|detiktv|grafis", re.I)  # photo/video/infographic pages have little text


def fetch(url, params=None, delay=1.5):
    """GET a page politely (pause after every request) and return it parsed."""
    r = requests.get(url, params=params, headers=HEADERS, timeout=20)
    time.sleep(delay)
    r.raise_for_status()
    return BeautifulSoup(r.text, "lxml")


def collect_urls(base, day, max_tries=4):
    """All article URLs a Detik channel listed on one day, e.g. base='https://finance.detik.com'."""
    pattern = re.compile(re.escape(base) + r"/([\w-]+)/d-(\d+)/[\w-]+")
    seen, urls = set(), {}
    page = last_page = 1
    while page <= last_page:
        for _ in range(max_tries):
            soup = fetch(f"{base}/indeks", params={"date": day.isoformat(), "page": page})
            links = [a["href"] for a in soup.select("a[href]")]
            ids = set(re.findall(r"/d-(\d+)/", " ".join(links)))
            if ids - seen:  # Detik sometimes returns a page we already have; if so, ask again
                break
        seen |= ids
        if page == 1:  # page 1 links to the day's last page ("1 2 3 4 5 ... 15")
            last_page = max([int(n) for h in links for n in re.findall(r"[?&]page=(\d+)", h)], default=1)
        for h in links:
            m = pattern.match(h)
            if m and not SKIP_SECTIONS.search(m.group(1)):
                urls[m.group(2)] = m.group(0)  # key = article id, so the same article isn't kept twice
        page += 1
    return list(urls.values())


def meta(soup, **attrs):
    tag = soup.find("meta", attrs=attrs)
    return tag.get("content") if tag else None


def parse_article(url):
    """Fetch one Detik article and return its fields (content = raw paragraphs, cleaned in 02_preprocessing)."""
    soup = fetch(url)
    body = soup.select_one("div.detail__body-text")
    paragraphs = [p.get_text(" ", strip=True) for p in body.find_all("p")] if body else []
    return {
        "url": url,
        "title": meta(soup, property="og:title"),
        "date": meta(soup, name="dtk:publishdate"),
        "author": meta(soup, name="dtk:author"),
        "tags": meta(soup, name="dtk:keywords"),
        "content": "\n".join(p for p in paragraphs if p),
    }

### G. Configuration of Labels and Channels

**Labels and channels** , plus:
- `START` / `END`: the date range to scrape 
- `PER_DAY`: articles to sample per label per day


In [3]:
CHANNELS = {
    "business":      ["https://finance.detik.com"],
    "technology":    ["https://inet.detik.com"],
    "sports":        ["https://sport.detik.com", "https://sport.detik.com/sepakbola"],
    "health":        ["https://health.detik.com"],
    "entertainment": ["https://hot.detik.com"],
}
LABELS = list(CHANNELS)

START = date(2026, 6, 1)   # provisional, confirmed in Section I
END = date(2026, 8, 31)    # provisional
PER_DAY = 14               # provisional
SEED = 42
LIMIT = None                  # days per label to run: 2 = smoke test, None = full range

RAW_DIR = Path("../data/raw")
RAW_DIR.mkdir(parents=True, exist_ok=True)
URLS_CSV = RAW_DIR / "urls.csv"
ARTICLES_CSV = RAW_DIR / "articles.csv"
FAILED_CSV = RAW_DIR / "failed.csv"

B_LIMIT = None             # articles to download this run: 20 = smoke test, None = all of urls.csv

# Stage A and Stage B are both finished: urls.csv holds 6,417 rows, articles.csv 6,417
# articles with 0 failures. While this flag is True the two scraping loops skip
# themselves, so an accidental Run All cannot re-download anything or append duplicates.
# Set it to False only to deliberately scrape again.
SCRAPING_DONE = True

### H. Pilot Test

Before scraping for hours, verify the functions work end to end on a single channel and a
single day: does `collect_urls` return a sensible number of URLs, does `parse_article`
return real article text, and does the stored text match what the website actually shows?

In [4]:
test_day = date(2026, 6, 3)
test_urls = collect_urls("https://health.detik.com", test_day)
print(len(test_urls), "article URLs found")

pilot = pd.DataFrame([parse_article(u) for u in test_urls[:3]])
pilot["words"] = pilot["content"].str.split().str.len()
pilot[["date", "title", "words", "url"]]

18 article URLs found


,date,title,words,url
0,2026/06/03 20:02:28,Indonesia Jangan Cuma Jadi Pasar Terapi Gen Du...,465,https://health.detik.com/berita-detikhealth/d-...
1,2026/06/03 19:05:17,"WMO Wanti-wanti Dampak El Nino di Indonesia, S...",539,https://health.detik.com/berita-detikhealth/d-...
2,2026/06/03 18:05:26,Kemenkes Ungkap Kebiasaan yang Kerap Tak Disad...,416,https://health.detik.com/berita-detikhealth/d-...


In [5]:
print(pilot.loc[0, "url"], "\n")
print(pilot.loc[0, "content"][:1000])

https://health.detik.com/berita-detikhealth/d-8516739/indonesia-jangan-cuma-jadi-pasar-terapi-gen-dunia-kepala-bpom-saatnya-jadi-pemain-utama 

Perkembangan terapi gen, terapi sel, hingga kecerdasan buatan (AI) tengah mengubah wajah dunia kesehatan global. Penyakit yang dulu sulit ditangani, seperti kanker, penyakit genetik langka, hingga kerusakan organ, kini mulai bisa diatasi melalui pendekatan yang lebih presisi dan berbasis teknologi biologis modern.
Kepala Badan Pengawas Obat dan Makanan (BPOM RI) Taruna Ikrar, menilai Indonesia tidak boleh hanya menjadi pasar bagi teknologi kesehatan yang dikembangkan negara lain. Menurutnya, Indonesia harus mulai mengambil peran sebagai pemain utama dalam inovasi biofarmasi modern.
"Transformasi besar sedang terjadi dalam dunia kesehatan global. Indonesia tidak boleh hanya menjadi pasar teknologi kesehatan dunia, tetapi harus mulai mengambil posisi sebagai pemain utama dalam inovasi biofarmasi modern," ujar Taruna dalam keterangannya.
ADVERTISE

### I. Volume Probe

`PER_DAY` must be set by the smallest channel, otherwise that label will fall short while the
others fill their quota. This probe counts available articles per label across 6 sample dates
(3 weekdays, 3 weekend days, spread over the intended period), since weekends publish less.
The results determine `PER_DAY`, `START` and `END` in Section G.

In [6]:
from statistics import median

# probe_dates = [date(2026, 6, 3),   # Wed
#                date(2026, 6, 6),   # Sat
#                date(2026, 6, 7),   # Sun
#                date(2026, 7, 15),  # Wed
#                date(2026, 8, 1),   # Sat
#                date(2026, 8, 19)]  # Wed

# probe = {}
# for label, bases in CHANNELS.items():
#     probe[label] = [len({u for b in bases for u in collect_urls(b, d)}) for d in probe_dates]
#     print(f"{label:14s} {probe[label]}  median={median(probe[label]):.0f}  min={min(probe[label])}")

### J. Stage A - Build the URL List

In this section, we'll start our plan on collecting articles from START to END (as we discussed on section G) from that label's
channel(s), remove duplicates by article ID, and randomly sample PER_DAY of them. Results are gonna be appended to 'urls.csv' after each label day fetched, so that we can resume if the process is interrupted.

In [7]:
# every date from START to END, inclusive
days = [START + timedelta(days=n) for n in range((END - START).days + 1)]

# the slice this run will cover - LIMIT is set in Section G
days_run = days if LIMIT is None else days[:LIMIT]

# (date, label) pairs already written, so a re-run can skip them
if URLS_CSV.exists() and URLS_CSV.stat().st_size > 0:
    prev = pd.read_csv(URLS_CSV, dtype=str)
    done = set(zip(prev["date"], prev["label"]))
else:
    done = set()

failures = []

# what is actually left to fetch, counted rather than assumed
todo = [(lbl, d) for lbl in LABELS for d in days_run
        if (d.isoformat(), lbl) not in done]

print(f"{len(days)} days in range ({days[0]} .. {days[-1]}), running {len(days_run)}")
print(f"label-days: {len(days_run) * len(LABELS)} total | done {len(done)} | to do {len(todo)}"
      f" | est. {len(todo) * 8 / 60:.0f} min at ~8 s each")


92 days in range (2026-06-01 .. 2026-08-31), running 92
label-days: 460 total | done 460 | to do 0 | est. 0 min at ~8 s each


In [8]:
days

[datetime.date(2026, 6, 1),
 datetime.date(2026, 6, 2),
 datetime.date(2026, 6, 3),
 datetime.date(2026, 6, 4),
 datetime.date(2026, 6, 5),
 datetime.date(2026, 6, 6),
 datetime.date(2026, 6, 7),
 datetime.date(2026, 6, 8),
 datetime.date(2026, 6, 9),
 datetime.date(2026, 6, 10),
 datetime.date(2026, 6, 11),
 datetime.date(2026, 6, 12),
 datetime.date(2026, 6, 13),
 datetime.date(2026, 6, 14),
 datetime.date(2026, 6, 15),
 datetime.date(2026, 6, 16),
 datetime.date(2026, 6, 17),
 datetime.date(2026, 6, 18),
 datetime.date(2026, 6, 19),
 datetime.date(2026, 6, 20),
 datetime.date(2026, 6, 21),
 datetime.date(2026, 6, 22),
 datetime.date(2026, 6, 23),
 datetime.date(2026, 6, 24),
 datetime.date(2026, 6, 25),
 datetime.date(2026, 6, 26),
 datetime.date(2026, 6, 27),
 datetime.date(2026, 6, 28),
 datetime.date(2026, 6, 29),
 datetime.date(2026, 6, 30),
 datetime.date(2026, 7, 1),
 datetime.date(2026, 7, 2),
 datetime.date(2026, 7, 3),
 datetime.date(2026, 7, 4),
 datetime.date(2026, 7, 5),

In [9]:
# GUARD - Stage A is finished: urls.csv holds 6,417 rows across all 460 label-days.
# While SCRAPING_DONE is True this empties days_run, so the loop below runs zero
# iterations: no requests, no writes. The loop itself is untouched and still readable.
# To scrape again on purpose: set SCRAPING_DONE = False in Section G, re-run Section G,
# then re-run the prep cell above so days_run is rebuilt.
if SCRAPING_DONE:
    print("SKIPPED: Stage A already complete - urls.csv has 6,417 rows. Nothing fetched.")
    days_run = []

for label in LABELS:
    for day in tqdm(days_run, desc=label):          # days_run honours LIMIT from Section G
        if (day.isoformat(), label) in done:        # string vs string - `done` was rebuilt from the CSV
            continue

        try:
            # 1. every URL this label's channel(s) listed that day
            pool = []
            for base in CHANNELS[label]:            # sports has two bases, so one base at a time
                pool += collect_urls(base, day)

            # 2. collapse duplicates by article id (same story filed under two sections)
            unique = {}
            for u in pool:
                m = re.search(r"/d-(\d+)/", u)
                if m:                               # a non-matching URL is skipped, not fatal
                    unique[m.group(1)] = u

            # 3. freeze the order, then sample reproducibly
            ordered = sorted(unique.values())
            n       = min(PER_DAY, len(ordered))    # thin days hold fewer than PER_DAY
            rng     = random.Random(f"{SEED}-{day}-{label}")
            picked  = rng.sample(ordered, n)

            # 4. append this label-day straight away
            rows   = [{"label": label, "date": day.isoformat(), "url": u} for u in picked]
            header = not URLS_CSV.exists()          # evaluated BEFORE the write -> header appears once
            pd.DataFrame(rows).to_csv(URLS_CSV, mode="a", header=header, index=False)

            # 5. only now is the label-day really done
            done.add((day.isoformat(), label))
            tqdm.write(f"{label} {day}: {len(pool)} found, {len(unique)} unique, {n} kept")

        except Exception as e:                      # not a bare `except:` - keeps Ctrl-C working
            failures.append((label, day, e))
            tqdm.write(f"FAILED {label} {day}: {type(e).__name__}: {e}")

print("label-days done:", len(done), "| failures:", len(failures))


SKIPPED: Stage A already complete - urls.csv has 6,417 rows. Nothing fetched.


business: 0it [00:00, ?it/s]
technology: 0it [00:00, ?it/s]
sports: 0it [00:00, ?it/s]
health: 0it [00:00, ?it/s]
entertainment: 0it [00:00, ?it/s]

label-days done: 460 | failures: 0


Result have been checked, resulting 6000+ urls.

### K. Stage B — Download the Articles

Stage A produced `urls.csv`, a list of 6,417 article URLs. Stage B walks that list, calls
`parse_article` on each URL, and appends the result to `articles.csv` — one row per article,
written immediately so an interrupted run keeps everything it already fetched.

At `delay=1.5` per request this is roughly 3.5 hours for the full list, so run it overnight.
`B_LIMIT` in Section G keeps the first pass small: 20 articles to prove the loop, then `None`
for the real run.

Resume works the same way as Stage A, but the key is the **URL** rather than the `(date, label)`
pair, because Stage B's unit of work is one article.

Content is stored **raw** — `SCROLL TO CONTINUE WITH CONTENT`, `ADVERTISEMENT` and the
`detikcom` self-mentions are left in and removed in `02_preprocessing`, so that cleaning
decisions are documented in the notebook that justifies them rather than buried in the scraper.


In [10]:
articles = pd.read_csv(URLS_CSV, dtype=str)
B_LIMIT = None             # articles to download this run: 20 = smoke test, None = all of urls.csv

# urls already downloaded, so a re-run can skip them
if ARTICLES_CSV.exists() and ARTICLES_CSV.stat().st_size > 0:
    prev = pd.read_csv(ARTICLES_CSV, dtype=str)
    got = set(prev["url"])
else:
    got = set()

# what is left to fetch, counted rather than assumed
work = articles[~articles["url"].isin(got)].reset_index(drop=True)
work = work if B_LIMIT is None else work.head(B_LIMIT)

failed_b = []


def fetch_and_append(url, label, index_date):
    """Download one article and append it to articles.csv.

    Returns None on success, or the error text so the caller can log it.
    Only parse_article is guarded: a failure to WRITE should stop the run,
    not be logged 6,000 times.
    """
    try:
        art = parse_article(url)
    except Exception as e:
        return f"{type(e).__name__}: {e}"

    art["label"] = label            # the channel label from urls.csv
    art["index_date"] = index_date  # the index date, kept beside the article's own publishdate

    header = not ARTICLES_CSV.exists()   # evaluated BEFORE the write -> header appears once
    pd.DataFrame([art]).to_csv(ARTICLES_CSV, mode="a", header=header, index=False)
    return None


print(f"urls.csv: {len(articles)} | already downloaded: {len(got)}"
      f" | to fetch now: {len(work)} of {len(articles) - len(got)} remaining")
print(f"est. {len(work) * 2 / 60:.0f} min at ~2 s each")


urls.csv: 6417 | already downloaded: 6417 | to fetch now: 0 of 0 remaining
est. 0 min at ~2 s each


In [11]:
# GUARD - Stage B is finished: articles.csv holds all 6,417 articles, 0 failures.
# While SCRAPING_DONE is True this empties the work list, so the loop runs zero
# iterations: no requests, no appends to articles.csv.
if SCRAPING_DONE:
    print("SKIPPED: Stage B already complete - articles.csv has 6,417 rows. Nothing downloaded.")
    work = work.head(0)

for r in tqdm(work.itertuples(index=False), total=len(work), desc="articles"):
    err = fetch_and_append(r.url, r.label, r.date)

    if err is None:
        got.add(r.url)                                   # only after the append succeeded
    else:
        failed_b.append({"url": r.url, "label": r.label, "date": r.date, "error": err})
        tqdm.write(f"FAILED {r.url}: {err}")

print("appended:", len(work) - len(failed_b), "| failed:", len(failed_b))


SKIPPED: Stage B already complete - articles.csv has 6,417 rows. Nothing downloaded.


articles: 0it [00:00, ?it/s]

appended: 0 | failed: 0


A failure here is usually a timeout or a 5xx, not a broken URL, so the failed list is worth
one retry before it is written to `failed.csv` for inspection. Run this cell only if
`failed_b` is non-empty.


In [12]:
still_failed = []

# GUARD - the real run finished with 0 failures, so there is nothing to retry. This also
# stops a stale failed_b from an older session sending requests on a re-run.
if SCRAPING_DONE:
    print("SKIPPED: nothing to retry - Stage B finished with 0 failures.")
    failed_b = []

for f in tqdm(failed_b, desc="retry"):
    err = fetch_and_append(f["url"], f["label"], f["date"])
    if err is None:
        got.add(f["url"])
    else:
        still_failed.append({**f, "error": err})

if still_failed:
    pd.DataFrame(still_failed).to_csv(FAILED_CSV, index=False)

print("recovered:", len(failed_b) - len(still_failed), "| still failed:", len(still_failed))


SKIPPED: nothing to retry - Stage B finished with 0 failures.


retry: 0it [00:00, ?it/s]

recovered: 0 | still failed: 0


Quick shape check on what landed. Empty and very short articles are **counted, not dropped** —
the minimum-length filter belongs in Task 2, where the threshold can be justified.


In [13]:
arts = pd.read_csv(ARTICLES_CSV, dtype=str)

print("rows:", len(arts))
print("columns:", arts.columns.tolist())
print(arts.groupby("label").size())

body = arts["content"].fillna("")
print("empty content:", (body.str.len() == 0).sum())
print("under 200 chars:", (body.str.len() < 200).sum())
print("median length:", int(body.str.len().median()))
print("duplicate urls:", arts["url"].duplicated().sum())
print("duplicate titles:", arts["title"].duplicated().sum())
print("mentions 'detik' in body:", body.str.contains("detik", case=False).mean().round(3))


rows: 6417
columns: ['url', 'title', 'date', 'author', 'tags', 'content', 'label', 'index_date']
label
business         1284
entertainment    1284
health           1283
sports           1282
technology       1284
dtype: int64
empty content: 3
under 200 chars: 525
median length: 2242
duplicate urls: 0
duplicate titles: 8
mentions 'detik' in body: 0.226


### L. Label Validation

The labels are **inherited from the channel**, not annotated — a `health.detik.com` article is
labelled `health` because of where it was published. That is a weak (proxy) label, and S2 sl.11
requires it to be validated rather than assumed.

So: draw a stratified sample of 120 articles (24 per label), read each one, and judge whether the
channel label actually fits the text. The disagreement rate that comes out of this is the honest
ceiling on what any classifier in Task 3 can be expected to reach — a model cannot be more correct
than its labels.

The cell below only **prepares** the sample and writes it to a CSV with two empty columns. The
judgement is filled in by hand.

In [14]:
PROC_DIR = Path("../data/processed")
PROC_DIR.mkdir(parents=True, exist_ok=True)
VALIDATION_CSV = PROC_DIR / "label_validation_sample.csv"

arts = pd.read_csv(ARTICLES_CSV, dtype=str)

# 24 per label, seeded so the sample is reproducible and can be cited
sample = (arts.groupby("label", group_keys=False)
              .sample(n=24, random_state=SEED)
              .reset_index(drop=True))

out = pd.DataFrame({
    "url": sample["url"],
    "channel_label": sample["label"],          # the label to be judged
    "title": sample["title"],
    "excerpt": sample["content"].fillna("").str.slice(0, 600),
    "label_fits": "",                          # fill by hand: yes / no / unsure
    "better_label": "",                         # fill by hand when label_fits = no
})

if VALIDATION_CSV.exists():
    print(f"{VALIDATION_CSV} already exists - NOT overwriting, so your judgements are safe.")
else:
    out.to_csv(VALIDATION_CSV, index=False)
    print(f"wrote {len(out)} rows to {VALIDATION_CSV}")

print(out.groupby("channel_label").size().to_dict())


..\data\processed\label_validation_sample.csv already exists - NOT overwriting, so your judgements are safe.
{'business': 24, 'entertainment': 24, 'health': 24, 'sports': 24, 'technology': 24}


Open `data/processed/label_validation_sample.csv`, read each excerpt, and fill `label_fits`
with `yes` / `no` / `unsure`. Where it is `no`, put what the label should have been in
`better_label`. Read the article itself when the excerpt is not enough — the url is in the row.

Then run the cell below to score it.

In [15]:
judged = pd.read_csv(VALIDATION_CSV, dtype=str).fillna("")
judged["label_fits"] = judged["label_fits"].str.strip().str.lower()

done_n = (judged["label_fits"] != "").sum()
print(f"judged: {done_n} / {len(judged)}")

if done_n == 0:
    print("Nothing judged yet - fill in the label_fits column first.")
else:
    print()
    print(judged["label_fits"].value_counts().to_dict())
    print()
    print("per label:")
    print(pd.crosstab(judged["channel_label"], judged["label_fits"]))

    agree = (judged["label_fits"] == "yes").sum()
    print()
    print(f"agreement rate: {agree}/{done_n} = {agree / done_n:.1%}")
    print(f"=> rough ceiling on Task 3 accuracy: ~{agree / done_n:.1%}")

    wrong = judged[judged["label_fits"] == "no"]
    if len(wrong):
        print()
        print("channel label -> label you would have given:")
        print(pd.crosstab(wrong["channel_label"], wrong["better_label"]))


judged: 120 / 120

{'yes': 105, 'no': 15}

per label:
label_fits     no  yes
channel_label         
business        4   20
entertainment   1   23
health          2   22
sports          1   23
technology      7   17

agreement rate: 105/120 = 87.5%
=> rough ceiling on Task 3 accuracy: ~87.5%

channel label -> label you would have given:
better_label      entertainment  sport  sports  technology  tragedy  unknown
channel_label                                                               
business       1              0      0       0           2        0        1
entertainment  0              0      0       0           0        1        0
health         0              0      1       0           0        0        1
sports         0              1      0       0           0        0        0
technology     3              0      0       1           0        0        3


### M. Data Quality Checks

Six checks against the Task 1 requirements, run on the finished corpus. Nothing is deleted here —
problems are **counted and recorded**, because the thresholds for dropping an article (minimum
length, which boilerplate to strip) belong in `02_preprocessing` where they can be justified.

In [16]:
arts = pd.read_csv(ARTICLES_CSV, dtype=str)
urls = pd.read_csv(URLS_CSV, dtype=str)

print("=== M1. Volume against the brief ===")
print(f"total articles: {len(arts)}   (brief requires >= 5,000)  ->",
      "PASS" if len(arts) >= 5000 else "FAIL")

per_label = arts.groupby("label").size()
print()
print(per_label.to_string())
print()
print(f"smallest label: {per_label.min()}   (floor is 1,000)  ->",
      "PASS" if per_label.min() >= 1000 else "FAIL")
print(f"downloaded {len(arts)} of {len(urls)} urls  ->",
      "complete" if len(arts) == len(urls) else f"{len(urls) - len(arts)} missing")


=== M1. Volume against the brief ===
total articles: 6417   (brief requires >= 5,000)  -> PASS

label
business         1284
entertainment    1284
health           1283
sports           1282
technology       1284

smallest label: 1282   (floor is 1,000)  -> PASS
downloaded 6417 of 6417 urls  -> complete


In [17]:
print("=== M2. Content quality ===")
body = arts["content"].fillna("")
chars = body.str.len()

print(f"empty content          : {(chars == 0).sum()}")
print(f"under 200 characters   : {(chars < 200).sum()}")
print(f"under 500 characters   : {(chars < 500).sum()}")
print(f"median characters      : {int(chars.median())}")
print(f"median words           : {int(body.str.split().str.len().median())}")
print()
print("length percentiles (characters):")
print(chars.describe(percentiles=[.01, .05, .25, .5, .75, .95, .99]).round(0).to_string())

print()
print("=== M3. Duplicates ===")
print(f"duplicate urls         : {arts['url'].duplicated().sum()}")
print(f"duplicate titles       : {arts['title'].duplicated().sum()}")
print(f"duplicate bodies       : {body[chars > 0].duplicated().sum()}")
print(f"null titles            : {arts['title'].isna().sum()}")

dup_titles = arts[arts["title"].duplicated(keep=False)].sort_values("title")
if len(dup_titles):
    print()
    print("articles sharing a title (check whether they are genuinely the same story):")
    print(dup_titles[["label", "index_date", "title"]].to_string(index=False, max_colwidth=70))


=== M2. Content quality ===
empty content          : 3
under 200 characters   : 525
under 500 characters   : 530
median characters      : 2242
median words           : 310

length percentiles (characters):
count     6417.0
mean      2393.0
std       1379.0
min          0.0
1%          90.0
5%         129.0
25%       1572.0
50%       2242.0
75%       3029.0
95%       4683.0
99%       7225.0
max      12757.0

=== M3. Duplicates ===
duplicate urls         : 0
duplicate titles       : 8
duplicate bodies       : 4
null titles            : 0

articles sharing a title (check whether they are genuinely the same story):
     label index_date                                                                  title
technology 2026-07-05     AC Split 1 PK Sharp Hemat Jutaan Rupiah di Transmart Full Day Sale
technology 2026-07-12     AC Split 1 PK Sharp Hemat Jutaan Rupiah di Transmart Full Day Sale
  business 2026-07-26            Alat Makan & Minum Banting Harga di Transmart Full Day Sale
  busines

In [18]:
print("=== M4. Date coverage ===")
month = arts["index_date"].str.slice(0, 7)
print(pd.crosstab(arts["label"], month).to_string())

print()
print(f"date range: {arts['index_date'].min()} .. {arts['index_date'].max()}")
print(f"distinct dates per label: {arts.groupby('label')['index_date'].nunique().to_dict()}")

print()
print("=== M5. Label leakage backlog for Task 2 ===")
patterns = {
    "SCROLL TO CONTINUE": "SCROLL TO CONTINUE",
    "ADVERTISEMENT": "ADVERTISEMENT",
    "detik (any form)": "detik",
    "detikcom": "detikcom",
    "Jakarta dateline": r"^Jakarta\s*-",
}
for name, pat in patterns.items():
    share = body.str.contains(pat, case=False, regex=True).mean()
    print(f"{name:22s} {share:6.1%}")

print()
print("channel self-mentions per label (these name the label itself):")
for lbl in sorted(arts["label"].unique()):
    sub = body[arts["label"] == lbl]
    print(f"  {lbl:14s} {sub.str.contains('detik', case=False).mean():6.1%}")


=== M4. Date coverage ===
index_date     2026-06  2026-07  2026-08
label                                   
business           416      434      434
entertainment      416      434      434
health             416      434      433
sports             416      432      434
technology         416      434      434

date range: 2026-06-01 .. 2026-08-31
distinct dates per label: {'business': 92, 'entertainment': 92, 'health': 92, 'sports': 92, 'technology': 92}

=== M5. Label leakage backlog for Task 2 ===
SCROLL TO CONTINUE      89.8%
ADVERTISEMENT           19.8%
detik (any form)        22.6%
detikcom                 5.9%
Jakarta dateline         8.1%

channel self-mentions per label (these name the label itself):
  business        24.1%
  entertainment    6.5%
  health          29.9%
  sports          13.0%
  technology      39.6%


In [19]:
import shutil
from datetime import datetime

print("=== M6. Ten random articles to eyeball against the live site ===")
spot = arts.sample(10, random_state=SEED)[["label", "index_date", "title", "url"]]
for r in spot.itertuples(index=False):
    print(f"[{r.label}] {r.index_date}  {r.title}")
    print(f"    {r.url}")

print()
print("=== Snapshot ===")
stamp = datetime.now().strftime("%Y%m%d")
snap = PROC_DIR / f"articles_snapshot_{stamp}.csv"
if snap.exists():
    print(f"{snap.name} already exists - not overwriting")
else:
    shutil.copy2(ARTICLES_CSV, snap)
    print(f"copied articles.csv -> {snap}")


=== M6. Ten random articles to eyeball against the live site ===
[sports] 2026-08-07  Madrid Gak Serius Kejar Rodri!
    https://sport.detik.com/sepakbola/liga-spanyol/d-8608945/madrid-gak-serius-kejar-rodri
[sports] 2026-07-29  Ni Kadek Dhinda Mulus di Kualifikasi Taipei Open 2026, Bidik Semifinal
    https://sport.detik.com/sport-lain/d-8594664/ni-kadek-dhinda-mulus-di-kualifikasi-taipei-open-2026-bidik-semifinal
[entertainment] 2026-07-17  Andy /rif Tak Paksa Anak Jadi Artis, Pilih Dukung Minat Balian dan Zilian
    https://hot.detik.com/celeb/d-8578116/andy-rif-tak-paksa-anak-jadi-artis-pilih-dukung-minat-balian-dan-zilian
[entertainment] 2026-08-28  Gaet Bemby Noor, Lulu Mudita Rilis Lagu Debut
    https://hot.detik.com/music/d-8638775/gaet-bemby-noor-lulu-mudita-rilis-lagu-debut
[health] 2026-06-06  Dilarang Prabowo Jadi Lauk MBG, Apa Sih Efeknya Jika Telur Dadar Dicampur Tepung?
    https://health.detik.com/diet/d-8520196/dilarang-prabowo-jadi-lauk-mbg-apa-sih-efeknya-jika-telur